In [17]:
import pandas as pd
from pathlib import Path
from IPython.display import display
from mdm.standardize import standardize_name, standardize_street, standardize_zip, extract_suite
from mdm.conform import one_table

path = Path("../data/sample")

def load(name):
    # Read one CSV, keeping every column as text
    return pd.read_csv(path / f"{name}.csv", dtype=str)

In [ ]:
header_names = ["record_uid", "source_system", "source_record_id", "role", "name_raw", "name_std", "street_std", "suite_std", "zip5", "legacy_customer_id", "npi"]
mapping = {
    "erp":{
        "record_uid": "erp_customer_id",
        "source_system": "erp",
        "role": "master",
        "name_raw": "customer_name",
        "street_std": "address_line_1",
        "suite_std": ["address_line_1", "address_line_2", "address_line_3"],
        "zip5": "postal_code",
        "legacy_customer_id": "legacy_customer_id",
        "npi": ""
    },
    "sf":{
        "record_uid": "sf_account_id",
        "source_system": "sf",
        "role": "crm",
        "name_raw": "account_name",
        "street_std": "billing_street",
        "suite_std": "billing_suite",
        "zip5": "billing_zip",
        "legacy_customer_id": "",
        "npi": "npi_number"        
    },
    "distributor":{
        "record_uid": "trace_id",
        "source_system": "distributor",
        "role": "tracing",
        "name_raw": "ship_to_name",
        "street_std": "ship_to_address",
        "suite_std": ["ship_to_address", "ship_to_name"],
        "zip5": "ship_to_zip",
        "legacy_customer_id": "",
        "npi": ""        
    }
}
df = pd.DataFrame(columns=header_names)

erp = load("erp_customer_master")
sf =  load("salesforce_accounts")
distributor_beta =  load("distributor_beta_tracing_2026-03")
distributor_alpha =  load("distributor_alpha_tracing_2026-03")

def one_table(df: pd.DataFrame, name: str, mapping: dict):

    if "beta" in name.lower():
        system_key = "beta"
    elif "distributor" in name.lower():
        system_key = "distributor"
    else:
        system_key = name
    
    if system_key not in mapping:
        raise ValueError(f"Unknown system '{name}'. Available: {list(mapping)}")
    system_branch = mapping[system_key]
    master_df = pd.DataFrame()

    master_df["record_uid"] = name + "::" + df[system_branch["record_uid"]] 
    master_df["source_system"] = name
    master_df["role"] = system_branch["role"]
    master_df["source_record_id"] = df[system_branch["record_uid"]]
    master_df["name_raw"] = df[system_branch["name_raw"]]
    master_df["name_std"] = df[system_branch["name_raw"]].apply(standardize_name)
    master_df["street_std"] = df[system_branch["street_std"]].apply(standardize_street)
    master_df["suite_std"] = df[system_branch["suite_std"]].apply(lambda row: extract_suite(*row), axis=1)
    master_df["zip5"] = df[system_branch["zip5"]].apply(lambda row: standardize_zip(row)[0])
    if system_branch["legacy_customer_id"] == "":
            ""
    else:
        master_df["legacy_customer_id"] = df[system_branch["legacy_customer_id"]]
    if system_branch["npi"] == "":
        ""
    else:
        master_df["npi"] = df[system_branch["npi"]]
    master_df = master_df.fillna("")
    return master_df.reindex(columns=header_names)

one_table(erp, "erp", mapping)

,record_uid,source_system,source_record_id,role,name_raw,name_std,street_std,suite_std,zip5,legacy_customer_id,npi
0,erp::E1001,erp,E1001,master,OAKMONT IMAGING CTR,OAKMONT IMAGING CENTER,505 ELM STREET,400,97205,L-0001,NaN
1,erp::E1002,erp,E1002,master,LAKEVIEW ORTHO,LAKEVIEW ORTHOPEDICS,505 ELM STREET,210,97205,L-0002,NaN
2,erp::E1003,erp,E1003,master,NORTHGATE REGIONAL HOSP,NORTHGATE REGIONAL HOSPITAL,88 CEDAR LANE,,97205,L-0003,NaN
3,erp::E1004,erp,E1004,master,BROOKSIDE SURGERY CTR,BROOKSIDE SURGERY CENTER,300 BIRCHWOOD COURT,,97205,L-0004,NaN
4,erp::E1005,erp,E1005,master,ST LUKE COMMUNITY HOSP,SAINT LUKE COMMUNITY HOSPITAL,505 PINE AVENUE,,97205,L-0005,NaN
...,...,...,...,...,...,...,...,...,...,...,...
58,erp::E1059,erp,E1059,master,SILVER CREEK HOSP,SILVER CREEK HOSPITAL,77 SOUTH ADAMS BOULEVARD,,50309,L-0061,NaN
59,erp::E1060,erp,E1060,master,BAYVIEW HOSP,BAYVIEW HOSPITAL,1240 COUNTY ROAD 9,,50309,L-0062,NaN
60,erp::E1061,erp,E1061,master,NORTHGATE SURGERY CTR,NORTHGATE SURGERY CENTER,505 SOUTH ADAMS BOULEVARD,,50309,L-0063,NaN
61,erp::E2003,erp,E2003,master,BROOKSIDE SURGERY CTR,BROOKSIDE SURGERY CENTER,300 BIRCHWOOD COURT,,97205,L-0004,NaN


In [ ]:
one_table(sf, "erp", mapping)

,record_uid,source_system,source_record_id,role,name_raw,name_std,street_std,suite_std,zip5,legacy_customer_id,npi
0,erp::E1001,erp,E1001,master,OAKMONT IMAGING CTR,OAKMONT IMAGING CENTER,505 ELM STREET,400,97205,L-0001,NaN
1,erp::E1002,erp,E1002,master,LAKEVIEW ORTHO,LAKEVIEW ORTHOPEDICS,505 ELM STREET,210,97205,L-0002,NaN
2,erp::E1003,erp,E1003,master,NORTHGATE REGIONAL HOSP,NORTHGATE REGIONAL HOSPITAL,88 CEDAR LANE,,97205,L-0003,NaN
3,erp::E1004,erp,E1004,master,BROOKSIDE SURGERY CTR,BROOKSIDE SURGERY CENTER,300 BIRCHWOOD COURT,,97205,L-0004,NaN
4,erp::E1005,erp,E1005,master,ST LUKE COMMUNITY HOSP,SAINT LUKE COMMUNITY HOSPITAL,505 PINE AVENUE,,97205,L-0005,NaN
...,...,...,...,...,...,...,...,...,...,...,...
58,erp::E1059,erp,E1059,master,SILVER CREEK HOSP,SILVER CREEK HOSPITAL,77 SOUTH ADAMS BOULEVARD,,50309,L-0061,NaN
59,erp::E1060,erp,E1060,master,BAYVIEW HOSP,BAYVIEW HOSPITAL,1240 COUNTY ROAD 9,,50309,L-0062,NaN
60,erp::E1061,erp,E1061,master,NORTHGATE SURGERY CTR,NORTHGATE SURGERY CENTER,505 SOUTH ADAMS BOULEVARD,,50309,L-0063,NaN
61,erp::E2003,erp,E2003,master,BROOKSIDE SURGERY CTR,BROOKSIDE SURGERY CENTER,300 BIRCHWOOD COURT,,97205,L-0004,NaN
